In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import BallTree
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
training_df = pd.read_csv(
    filepath_or_buffer='../data/training_faults_diagnostics.csv',
    low_memory=False
)
training_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1058069 entries, 0 to 1058068
Data columns (total 48 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   RecordID                   1058069 non-null  int64  
 1   EventTimeStamp             1058069 non-null  object 
 2   eventDescription           1002335 non-null  object 
 3   ecuSoftwareVersion         831493 non-null   object 
 4   ecuModel                   1002466 non-null  object 
 5   ecuMake                    1002466 non-null  object 
 6   ecuSource                  1058069 non-null  int64  
 7   spn                        1058069 non-null  int64  
 8   fmi                        1058069 non-null  int64  
 9   active                     1058069 non-null  bool   
 10  activeTransitionCount      1058069 non-null  int64  
 11  EquipmentID                1058069 non-null  object 
 12  MCTNumber                  1058069 non-null  int64  
 13  Latitude    

In [3]:
# Drop columns that aren't needed for the model or aren't working witht the pipeline
training_df = training_df.drop(columns=[
    'RecordID',
    'EventTimeStamp',
    'LocationTimeStamp',
    'eventDescription',
    'ecuSoftwareVersion',
    'ecuModel',
    'ecuMake',
    'ecuSource',
    'EquipmentID',
    'NearServiceStation',
    'IsFullDerate',
    'Severity_Level',
    'Derate_Target_2.0-0.001',
    'Derate_Target_4.0-0.001',
    'Derate_Target_8.0-0.001'
])

## Identify features for imputing missing values

In [4]:
# Drop columns that have too many NaN values
nan_drop_threshold = 0.8

drop_columns = training_df.columns[training_df.isna().mean() > nan_drop_threshold]
print(drop_columns)

training_df = training_df.drop(columns=drop_columns)

Index(['ServiceDistance', 'SwitchedBatteryVoltage'], dtype='object')


In [5]:
# Group categorical columns
categorical_columns = training_df.select_dtypes(include=["object", "bool"]).columns
categorical_columns

Index(['active', 'CruiseControlActive', 'IgnStatus', 'ParkingBrake'], dtype='object')

In [6]:
# Group numeric columns
numeric_columns = training_df.select_dtypes(include=["int64", "float64"]).columns
numeric_columns

Index(['spn', 'fmi', 'activeTransitionCount', 'MCTNumber', 'Latitude',
       'Longitude', 'Severity_Level_Numeric', 'Derate_Target_12.0-0.001',
       'AcceleratorPedal', 'BarometricPressure', 'CruiseControlSetSpeed',
       'DistanceLtd', 'EngineCoolantTemperature', 'EngineLoad',
       'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm',
       'EngineTimeLtd', 'FuelLevel', 'FuelLtd', 'FuelRate', 'FuelTemperature',
       'IntakeManifoldTemperature', 'LampStatus', 'Speed', 'Throttle',
       'TurboBoostPressure'],
      dtype='object')

In [7]:
# Group numeric columns by threshold
nan_low_threshold = 0.4

low_nan_numeric_columns = training_df[numeric_columns].columns[
    training_df[numeric_columns].isna().mean() <= nan_low_threshold
]
print(low_nan_numeric_columns)

medium_nan_numeric_columns = training_df[numeric_columns].columns[
    training_df[numeric_columns].isna().mean() > nan_low_threshold
]
print(medium_nan_numeric_columns)

Index(['spn', 'fmi', 'activeTransitionCount', 'MCTNumber', 'Latitude',
       'Longitude', 'Derate_Target_12.0-0.001', 'LampStatus'],
      dtype='object')
Index(['Severity_Level_Numeric', 'AcceleratorPedal', 'BarometricPressure',
       'CruiseControlSetSpeed', 'DistanceLtd', 'EngineCoolantTemperature',
       'EngineLoad', 'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm',
       'EngineTimeLtd', 'FuelLevel', 'FuelLtd', 'FuelRate', 'FuelTemperature',
       'IntakeManifoldTemperature', 'Speed', 'Throttle', 'TurboBoostPressure'],
      dtype='object')


## Split training dataset

In [8]:
X = training_df.drop(columns=['Derate_Target_12.0-0.001'])
y = training_df['Derate_Target_12.0-0.001']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

## Create pipeline and fit model

In [10]:
categorical_pipe = Pipeline(
    steps=[
        ('categorical_imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder())
    ]
)

low_nan_numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('low_nan_numeric_imputer', SimpleImputer(strategy='median'))
    ]
)

medium_nan_numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('medium_nan_numeric_imputer', IterativeImputer(max_iter=20, random_state=30))
    ]
)

In [11]:
ct = ColumnTransformer(
    transformers=[
        ('categorical_pipe', categorical_pipe, categorical_columns),
        ('low_nan_numeric_pipe', low_nan_numeric_pipe, low_nan_numeric_columns.drop('Derate_Target_12.0-0.001')),
        ('medium_nan_numeric_pipe', medium_nan_numeric_pipe, medium_nan_numeric_columns)
    ]
)

In [12]:
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('model', MLPClassifier(hidden_layer_sizes=(2,)))
    ]
)

In [13]:
pipe.fit(X_train, y_train)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\impute\_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


,steps,"[('transformer', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categorical_pipe', ...), ('low_nan_numeric_pipe', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [14]:
y_pred = pipe.predict(X_test)

In [15]:
confusion_matrix(
    y_true=y_test,
    y_pred=y_pred
)

array([[316865,      0],
       [   556,      0]])

In [16]:
print(classification_report(
    y_true=y_test,
    y_pred=y_pred
))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    316865
           1       0.00      0.00      0.00       556

    accuracy                           1.00    317421
   macro avg       0.50      0.50      0.50    317421
weighted avg       1.00      1.00      1.00    317421



C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
